# 2025

## Full Pipeline

In [ ]:
# import cv2
# import numpy as np
# import os
# import glob

# class BorobudurPreprocessor:
#     def __init__(self, input_folder, output_folder):
#         self.input_folder = input_folder
#         self.output_folder = output_folder
#         if not os.path.exists(output_folder):
#             os.makedirs(output_folder)

#     def stitch_images(self, image_paths):
#         """
#         Menggabungkan array gambar menjadi satu panorama.
#         Menggunakan algoritma SIFT untuk fitur dan RANSAC untuk Homography.
#         """
#         print(f"--- Memulai Stitching untuk {len(image_paths)} gambar... ---")
        
#         # 1. Baca gambar pertama sebagai base
#         base_img = cv2.imread(image_paths[0])
        
#         # Loop untuk menggabungkan gambar berikutnya satu per satu
#         for i in range(1, len(image_paths)):
#             next_img = cv2.imread(image_paths[i])
            
#             # Inisialisasi SIFT detector
#             sift = cv2.SIFT_create()
            
#             # Temukan Keypoints dan Descriptors
#             kp1, des1 = sift.detectAndCompute(base_img, None)
#             kp2, des2 = sift.detectAndCompute(next_img, None)
            
#             # Feature Matching menggunakan FLANN
#             FLANN_INDEX_KDTREE = 1
#             index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
#             search_params = dict(checks=50)
#             flann = cv2.FlannBasedMatcher(index_params, search_params)
#             matches = flann.knnMatch(des1, des2, k=2)
            
#             # Filter match yang bagus (Lowe's ratio test)
#             good_matches = []
#             for m, n in matches:
#                 if m.distance < 0.7 * n.distance:
#                     good_matches.append(m)
            
#             # Butuh minimal 4 titik untuk hitung Homography
#             if len(good_matches) > 4:
#                 src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
#                 dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
                
#                 # Hitung Matrix Homography
#                 H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)
                
#                 # Warp image (Melengkungkan gambar kedua agar pas dengan gambar pertama)
#                 height, width = base_img.shape[:2]
#                 warped_next = cv2.warpPerspective(next_img, H, (width + next_img.shape[1], height))
                
#                 # Blending Sederhana (Menempelkan)
#                 # Catatan: Untuk hasil sempurna, perlu Multi-band Blending, tapi ini versi dasar
#                 warped_next[0:height, 0:width] = base_img
#                 base_img = warped_next
                
#                 # Crop area hitam sisa warping (Opsional, penyederhanaan)
#                 # base_img = self._crop_black_borders(base_img) 
                
#             else:
#                 print(f"Gagal stitch gambar ke-{i}. Kurang titik match.")
                
#         print("Stitching Selesai.")
#         return base_img

#     def convert_to_grayscale(self, image):
#         """Mengubah BGR ke Grayscale"""
#         return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

#     def normalize_image(self, image):
#         """
#         Normalisasi piksel dari 0-255 (int) ke 0.0-1.0 (float32).
#         Penting untuk input Neural Network.
#         """
#         img_float = image.astype(np.float32)
#         img_normalized = img_float / 255.0
#         return img_normalized

#     def save_processed_data(self, image_norm, original_name):
#         """Menyimpan hasil. Karena float 0-1 tidak bisa dilihat lgsg sbg PNG, kita simpan sbg .npy (data) dan .png (preview)"""
#         base_name = os.path.splitext(original_name)[0]
        
#         # 1. Simpan sebagai Data NumPy (Untuk Training nanti) -> Presisi Tinggi
#         np.save(os.path.join(self.output_folder, base_name + '_normalized.npy'), image_norm)
        
#         # 2. Simpan sebagai PNG Preview (Kembalikan ke 0-255 utk visualisasi)
#         preview_img = (image_norm * 255).astype(np.uint8)
#         cv2.imwrite(os.path.join(self.output_folder, base_name + '_preview.png'), preview_img)
#         print(f"Data tersimpan: {base_name}")

#     def run_pipeline(self):
#         # Anggap struktur foldernya: input_folder/panel_1/foto1.jpg, foto2.jpg...
#         # Untuk demo ini, kita ambil semua jpg di folder input
#         images = sorted(glob.glob(os.path.join(self.input_folder, "*.jpg")))
        
#         if not images:
#             print("Tidak ada gambar ditemukan.")
#             return

#         # 1. Stitching
#         stitched_img = self.stitch_images(images)
        
#         # 2. Grayscale
#         gray_img = self.convert_to_grayscale(stitched_img)
        
#         # 3. Normalization
#         norm_img = self.normalize_image(gray_img)
        
#         # 4. Save
#         self.save_processed_data(norm_img, "panel_gabungan_final.jpg")

# # --- CARA JALANIN KODE ---
# if __name__ == "__main__":
#     # Ganti path sesuai komputer Anda
#     input_path = "dataset/raw_images/panel_01" 
#     output_path = "dataset/processed_2025/panel_01"
    
#     processor = BorobudurPreprocessor(input_path, output_path)
#     processor.run_pipeline()

## Image Stitching

In [1]:
import cv2
import numpy as np
import os
import glob
import re

# --- FUNGSI UTAMA STITCHING ---
def stitch_pair(img1, img2, max_retries=5):
    """
    Mencoba menjahit img2 ke img1.
    Melakukan retry hingga max_retries jika gagal menemukan Homography yang valid.
    """
    attempt = 0
    sift = cv2.SIFT_create()

    while attempt < max_retries:
        try:
            # 1. Deteksi Keypoint & Deskriptor
            kp1, des1 = sift.detectAndCompute(img1, None)
            kp2, des2 = sift.detectAndCompute(img2, None)

            # Cek jika deskriptor kosong
            if des1 is None or des2 is None:
                print(f"   [!] Percobaan {attempt+1}: Tidak ada fitur terdeteksi.")
                attempt += 1
                continue

            # 2. Matching Fitur (FLANN)
            FLANN_INDEX_KDTREE = 1
            index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
            search_params = dict(checks=50)
            flann = cv2.FlannBasedMatcher(index_params, search_params)
            matches = flann.knnMatch(des1, des2, k=2)

            # 3. Filter Good Matches (Lowe's Ratio Test)
            good_matches = []
            for m, n in matches:
                if m.distance < 0.7 * n.distance:
                    good_matches.append(m)

            # 4. Hitung Homography (Butuh minimal 4 titik)
            if len(good_matches) > 4:
                src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
                dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

                # RANSAC adalah proses acak, jadi retry bisa memberikan hasil beda
                H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)
                
                if H is None:
                    raise Exception("Homography Matrix kosong")

                # 5. Warping & Blending
                height, width = img1.shape[:2]
                # Hitung ukuran kanvas baru (sederhana: lebar digabung)
                warped_img2 = cv2.warpPerspective(img2, H, (width + img2.shape[1], height))
                
                # Tempel img1 ke hasil warp
                warped_img2[0:height, 0:width] = img1
                
                # Potong area hitam berlebih (Simple cropping)
                # Mencari kolom terakhir yang tidak hitam total untuk crop
                gray = cv2.cvtColor(warped_img2, cv2.COLOR_BGR2GRAY)
                _, thresh = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
                contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if contours:
                    x, y, w, h = cv2.boundingRect(contours[0])
                    warped_img2 = warped_img2[y:y+h, x:x+w]

                return warped_img2 # Sukses

            else:
                print(f"   [!] Percobaan {attempt+1}: Kurang titik match ({len(good_matches)} < 4).")

        except Exception as e:
            print(f"   [!] Percobaan {attempt+1} Gagal: {e}")

        attempt += 1

    # Jika loop selesai tanpa return, berarti gagal total
    return None

def process_stitching_folder(input_folder):
    """
    Otomatis mendeteksi grup (misal 16a, 16b -> Grup 16) dan men-stitch.
    """
    # 1. Kelompokkan File
    # Format dictionary: { '16': ['path/16a.png', 'path/16b.png'], ... }
    groups = {}
    all_files = sorted(glob.glob(os.path.join(input_folder, "*.png"))) # Asumsi PNG sesuai request
    
    print(f"Total file ditemukan: {len(all_files)}")

    for filepath in all_files:
        filename = os.path.basename(filepath)
        # Regex untuk mengambil angka di depan (misal: "16" dari "16a.png")
        match = re.match(r"(\d+)", filename)
        if match:
            group_id = match.group(1)
            if group_id not in groups:
                groups[group_id] = []
            groups[group_id].append(filepath)
    
    # Buat subfolder output untuk hasil stitch (agar tidak campur dengan raw)
    output_stitch_folder = os.path.join(input_folder, "stitched_raw")
    if not os.path.exists(output_stitch_folder):
        os.makedirs(output_stitch_folder)

    # 2. Proses per Grup
    for group_id, file_list in groups.items():
        print(f"\n--- Memproses Panel Nomor {group_id} ({len(file_list)} gambar) ---")
        
        # Urutkan a, b, c...
        file_list.sort() 
        
        if len(file_list) < 2:
            print(f"   [INFO] Hanya 1 gambar, menyalin langsung...")
            base_img = cv2.imread(file_list[0])
        else:
            # Stitch gambar pertama dengan kedua, hasilnya stitch dengan ketiga, dst.
            base_img = cv2.imread(file_list[0])
            
            for i in range(1, len(file_list)):
                next_img_path = file_list[i]
                print(f"   > Menjahit {os.path.basename(next_img_path)} ke hasil sementara...")
                next_img = cv2.imread(next_img_path)
                
                result = stitch_pair(base_img, next_img, max_retries=5)
                
                if result is not None:
                    base_img = result
                else:
                    print(f"   [ERROR] Gagal menjahit {os.path.basename(next_img_path)} setelah 5x percobaan. SKIP sisa panel ini.")
                    base_img = None # Tandai gagal
                    break 
        
        # 3. Simpan Hasil
        if base_img is not None:
            save_path = os.path.join(output_stitch_folder, f"{group_id}.png")
            cv2.imwrite(save_path, base_img)
            print(f"   [SUKSES] Panel {group_id} tersimpan di: {save_path}")
        else:
            print(f"   [GAGAL] Panel {group_id} dilewati.")

# --- JALANKAN CELL 1 ---
# Ganti path di bawah ini sesuai lokasi folder Anda
input_folder_path = "dataset/Complete/Partial/PNG" 
process_stitching_folder(input_folder_path)

Total file ditemukan: 38

--- Memproses Panel Nomor 16 (3 gambar) ---
   > Menjahit 16b.png ke hasil sementara...
   > Menjahit 16c.png ke hasil sementara...
   [SUKSES] Panel 16 tersimpan di: dataset/Complete/Partial/PNG\stitched_raw\16.png

--- Memproses Panel Nomor 17 (3 gambar) ---
   > Menjahit 17b.png ke hasil sementara...
   > Menjahit 17c.png ke hasil sementara...
   [!] Percobaan 1: Tidak ada fitur terdeteksi.
   [!] Percobaan 2: Tidak ada fitur terdeteksi.
   [!] Percobaan 3: Tidak ada fitur terdeteksi.
   [!] Percobaan 4: Tidak ada fitur terdeteksi.
   [!] Percobaan 5: Tidak ada fitur terdeteksi.
   [ERROR] Gagal menjahit 17c.png setelah 5x percobaan. SKIP sisa panel ini.
   [GAGAL] Panel 17 dilewati.

--- Memproses Panel Nomor 18 (3 gambar) ---
   > Menjahit 18b.png ke hasil sementara...
   > Menjahit 18c.png ke hasil sementara...
   [SUKSES] Panel 18 tersimpan di: dataset/Complete/Partial/PNG\stitched_raw\18.png

--- Memproses Panel Nomor 21 (2 gambar) ---
   > Menjahit 21

KeyboardInterrupt: 

## Preprocessing

In [ ]:
import cv2
import numpy as np
import os
import glob

def preprocess_and_save(input_folder):
    """
    Mengambil gambar hasil stitch, ubah ke Grayscale, Normalisasi 0-1,
    dan simpan sebagai .npy (data) dan .png (preview).
    """
    # Folder input adalah output dari proses sebelumnya
    source_folder = os.path.join(input_folder, "stitched_raw")
    
    # Folder output final
    output_folder = os.path.join(input_folder, "processed")
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    files = glob.glob(os.path.join(source_folder, "*.png"))
    print(f"Ditemukan {len(files)} gambar hasil stitch untuk diproses.")
    
    for filepath in files:
        filename = os.path.basename(filepath)
        name_no_ext = os.path.splitext(filename)[0] # misal "16"
        
        print(f"Processing: {filename} ...")
        
        # 1. Baca Gambar
        img = cv2.imread(filepath)
        if img is None:
            print(f"   [ERROR] Tidak bisa membaca {filepath}")
            continue
            
        # 2. Konversi Grayscale
        gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # 3. Normalisasi (0-255 -> 0.0-1.0)
        # Mengubah tipe data menjadi float32 agar presisi
        norm_img = gray_img.astype(np.float32) / 255.0
        
        # 4. Simpan Data
        
        # A. Simpan sebagai file NumPy (.npy) -> INI YANG DIPAKAI TRAIN MODEL
        # File ini menyimpan nilai float presisi tinggi (0.123...)
        npy_path = os.path.join(output_folder, f"{name_no_ext}.npy")
        np.save(npy_path, norm_img)
        
        # B. Simpan sebagai PNG (.png) -> HANYA UNTUK PREVIEW MATA MANUSIA
        # Kita kembalikan ke 0-255 uint8 hanya untuk disimpan sebagai gambar
        preview_img = (norm_img * 255).astype(np.uint8)
        png_path = os.path.join(output_folder, f"{name_no_ext}_bw_norm.png")
        cv2.imwrite(png_path, preview_img)
        
        print(f"   -> Disimpan: {npy_path}")

    print("\n--- Semua Proses Selesai ---")
    print(f"Cek folder output di: {output_folder}")

# --- JALANKAN CELL 2 ---
# Pastikan path ini SAMA dengan input_folder_path di Cell 1
input_folder_path = "dataset/raw_images" 
preprocess_and_save(input_folder_path)

# 2026

## Ratio Calculator

In [ ]:
import os
import glob
import numpy as np
import math

class BorobudurRatioCalculator:
    def __init__(self, annotation_folder, img_width, img_height):
        self.folder = annotation_folder
        self.W = img_width
        self.H = img_height
        
        # --- KONFIGURASI ID KELAS (SESUAIKAN DENGAN FILE classes.txt ANDA) ---
        self.ID_LENGAN_BAWAH = 0  # Ganti dengan ID class Lengan Bawah di dataset Anda
        self.ID_TANGAN = 1        # Ganti dengan ID class Telapak Tangan di dataset Anda
        
        self.ratios = []

    def denormalize_polygon(self, coords):
        """Mengubah koordinat normal (0-1) menjadi piksel nyata (W, H)."""
        # Format YOLO Seg: x1, y1, x2, y2, ...
        # Kita ubah jadi list of points: [[x1, y1], [x2, y2], ...]
        points = []
        for i in range(0, len(coords), 2):
            x = float(coords[i]) * self.W
            y = float(coords[i+1]) * self.H
            points.append([x, y])
        return np.array(points)

    def get_max_length(self, polygon_points):
        """
        Mengukur panjang organ dengan mencari jarak terjauh antara dua titik 
        dalam poligon (Diameter Geodesik Sederhana).
        Cocok untuk bentuk memanjang seperti lengan.
        """
        if len(polygon_points) < 2:
            return 0
        
        # Hitung jarak antar semua pasangan titik dalam poligon
        # Ini pendekatan brute-force yang akurat untuk jumlah titik sedikit (<100)
        # Untuk optimasi bisa pakai Convex Hull, tapi ini cukup cepat.
        max_dist = 0
        for i in range(len(polygon_points)):
            for j in range(i + 1, len(polygon_points)):
                p1 = polygon_points[i]
                p2 = polygon_points[j]
                dist = np.linalg.norm(p1 - p2)
                if dist > max_dist:
                    max_dist = dist
        return max_dist

    def get_centroid(self, polygon_points):
        """Mencari titik tengah poligon untuk pairing."""
        return np.mean(polygon_points, axis=0)

    def process_files(self):
        txt_files = glob.glob(os.path.join(self.folder, "*.txt"))
        print(f"Ditemukan {len(txt_files)} file anotasi.")

        for file_path in txt_files:
            with open(file_path, 'r') as f:
                lines = f.readlines()

            lengan_list = []
            tangan_list = []

            # 1. Parsing setiap baris dalam satu file
            for line in lines:
                parts = line.strip().split()
                class_id = int(parts[0])
                coords = parts[1:] # Ambil koordinat poligon

                poly_pixels = self.denormalize_polygon(coords)
                length = self.get_max_length(poly_pixels)
                centroid = self.get_centroid(poly_pixels)

                obj_data = {
                    'centroid': centroid,
                    'length': length
                }

                if class_id == self.ID_LENGAN_BAWAH:
                    lengan_list.append(obj_data)
                elif class_id == self.ID_TANGAN:
                    tangan_list.append(obj_data)

            # 2. Pairing: Pasangkan Lengan dengan Tangan terdekat
            # Kita loop Lengan, cari Tangan yang centroid-nya paling dekat
            used_tangan_indices = set()

            for lengan in lengan_list:
                best_match_idx = -1
                min_dist = float('inf')

                for idx, tangan in enumerate(tangan_list):
                    if idx in used_tangan_indices:
                        continue # Tangan ini sudah punya pasangan
                    
                    # Hitung jarak antar centroid
                    dist = np.linalg.norm(lengan['centroid'] - tangan['centroid'])
                    
                    # Ambang batas jarak (misal: jangan pasangkan kalau terlalu jauh/beda tokoh)
                    # Bisa disesuaikan, misal max 300 piksel
                    if dist < min_dist:
                        min_dist = dist
                        best_match_idx = idx

                if best_match_idx != -1:
                    # Pasangan ditemukan!
                    match_tangan = tangan_list[best_match_idx]
                    used_tangan_indices.add(best_match_idx)
                    
                    # HITUNG RASIO
                    # Rumus: Panjang Lengan Bawah / Panjang Tangan
                    if match_tangan['length'] > 0:
                        ratio = lengan['length'] / match_tangan['length']
                        self.ratios.append(ratio)

    def calculate_statistics(self):
        if not self.ratios:
            print("Tidak ada pasangan Lengan-Tangan yang ditemukan.")
            return

        data = np.array(self.ratios)
        print(f"\n--- Statistik Awal (N={len(data)}) ---")
        print(f"Mean: {np.mean(data):.4f}")
        print(f"Min: {np.min(data):.4f}, Max: {np.max(data):.4f}")

        # 3. Hapus Outlier menggunakan IQR (Interquartile Range)
        Q1 = np.percentile(data, 25)
        Q3 = np.percentile(data, 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        clean_data = data[(data >= lower_bound) & (data <= upper_bound)]
        
        print(f"\n--- Statistik Valid (Setelah Hapus Outlier) (N={len(clean_data)}) ---")
        print(f"Outlier dihapus: {len(data) - len(clean_data)}")
        
        final_mean = np.mean(clean_data)
        final_median = np.median(clean_data)
        
        print(f"RATA-RATA RASIO (MEAN): {final_mean:.4f}")
        print(f"MEDIAN RASIO: {final_median:.4f}")
        print("-" * 30)
        print(f"KESIMPULAN: Panjang Lengan Bawah adalah {final_mean:.2f} kali Panjang Tangan.")

# --- BAGIAN EKSEKUSI UTAMA ---
if __name__ == "__main__":
    # Ganti path ini ke folder dataset Anda (folder 'train/labels' atau sejenisnya)
    FOLDER_PATH = "dataset_borobudur/labels/train" 
    
    # Masukkan resolusi rata-rata atau resolusi gambar panel Anda
    # Ini penting agar konversi piksel akurat
    IMG_W = 1920 
    IMG_H = 1080

    calculator = BorobudurRatioCalculator(FOLDER_PATH, IMG_W, IMG_H)
    
    # Setting ID Class (Pastikan ini sesuai file classes.txt Anda)
    calculator.ID_LENGAN_BAWAH = 0 
    calculator.ID_TANGAN = 1       

    print("Sedang memproses...")
    calculator.process_files()
    calculator.calculate_statistics()